# Análisis y estimación de la duración de causas judiciales
## Tercera entrega: entrenamiento y validación
**Integrantes:** Brian Barbizan y Daniela Palazzo  
**Materia:** Práctica Profesional: Análisis y Exploración de Datos

Trabajamos con causas civiles de la provincia de Buenos Aires. Buscamos estimar su duración total en días corridos a partir de la localidad, el juzgado y el objeto del litigio. Esta etapa reúne el entrenamiento, la evaluación y el guardado del modelo para su integración con la API.

Usamos regresión porque la variable que queremos estimar es numérica. El alcance es histórico: no estamos estimando el tiempo restante de un expediente abierto ni garantizando plazos actuales.

**Ejecución:**
- Colocar esta notebook y el archivo "causas_limpias_buenos_aires.csv" en la misma carpeta
- En Colab, subir el CSV al entorno. Ejecutar las celdas en orden

Las versiones utilizadas están en el archivo "requirements.txt"

## 1. Librerías y preparación del entorno
Fijamos las versiones de las librerías y usamos random_state=42 para poder repetir el entrenamiento y reproducir los resultados

In [1]:
import json
import platform

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
)
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


BASE_DIR = Path.cwd()

VARIABLES = [
    "localidad_nombre",
    "juzgado_numero",
    "causa_objeto_litigio_descripcion",
]

CLAVE_CAUSA = [
    "provincia_id",
    "localidad_nombre",
    "juzgado_numero",
    "causa_id",
]

OBJETIVO = "duracion_dias"


def crear_pipeline(estimador):
    preparacion = ColumnTransformer(
        [
            (
                "categorias",
                OneHotEncoder(
                    handle_unknown="ignore",
                ),
                VARIABLES,
            )
        ]
    )

    return Pipeline(
        [
            ("preparacion", preparacion),
            ("modelo", estimador),
        ]
    )



print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)

Python: 3.13.18
scikit-learn: 1.6.1


## 2. Datos y controles previos
Partimos del CSV curado en la etapa anterior. Fuente: Ministerio de Justicia, dataset Oralidad en los procesos civiles: https://datos.jus.gob.ar/dataset/oralidad-en-los-procesos-civiles. La duración se calculó entre recepción y finalización. Comprobamos columnas, faltantes, duplicados por causa y duraciones inválidas. Los valores atípicos válidos se conservan.

Las fechas de finalización y los identificadores se usan para controlar y separar los datos; no se incorporan como predictores. Tampoco usamos variables conocidas únicamente al cierre. La muestra excluye causas sin cierre informado, lo que puede sesgar los resultados

In [3]:
origen = (
    BASE_DIR
    / "causas_limpias_buenos_aires.csv"
)

datos = pd.read_csv(
    origen,
    dtype="string",
    encoding="utf-8-sig",
)

requeridas = set(
    VARIABLES
    + CLAVE_CAUSA
    + ["fecha_fin", OBJETIVO]
)

faltantes = requeridas - set(datos.columns)

if faltantes:
    raise ValueError(
        f"Faltan columnas: {faltantes}"
    )

if datos[list(requeridas)].isna().any().any():
    raise ValueError(
        "Hay valores faltantes. "
        "Revisá la preparación de datos."
    )

if datos.duplicated(CLAVE_CAUSA).any():
    raise ValueError(
        "Hay causas duplicadas."
    )

datos[OBJETIVO] = pd.to_numeric(
    datos[OBJETIVO],
    errors="raise",
)

if (
    not np.isfinite(datos[OBJETIVO]).all()
    or (datos[OBJETIVO] < 0).any()
):
    raise ValueError(
        "Hay duraciones inválidas."
    )

datos["fecha_fin"] = pd.to_datetime(
    datos["fecha_fin"],
    format="%Y-%m-%d",
    errors="raise",
)

datos = (
    datos
    .sort_values(
        ["fecha_fin"] + CLAVE_CAUSA
    )
    .reset_index(drop=True)
)

if len(datos) < 100:
    raise ValueError(
        "Hay muy pocas causas para esta evaluación."
    )



In [4]:
print("Causas:", len(datos))
print(datos[VARIABLES + [OBJETIVO]].head().to_string(index=False))
print(datos[OBJETIVO].describe().to_string())

Causas: 8752
        localidad_nombre juzgado_numero     causa_objeto_litigio_descripcion  duracion_dias
                LA PLATA              5                  AMPARO (INFOREC 10)              0
MORENO-GENERAL RODRIGUEZ              1                  AMPARO (INFOREC 10)              1
                 DOLORES              2               DESALOJO FALTA DE PAGO            126
               SAN JUSTO              1                  AMPARO (INFOREC 10)              3
              SAN MARTIN              9 DESALOJO (EXCEPTO POR FALTA DE PAGO)            357
count        8752.0
mean     790.997258
std      539.053882
min             0.0
25%           420.0
50%           672.0
75%         1002.25
max          5059.0


## 3. Separación temporal
Ordenamos por fecha de finalización y definimos cortes aproximados de 60 % para entrenamiento, 20 % para validación y 20 % para prueba. Una misma fecha no queda repartida entre conjuntos. Esta separación evalúa cierres posteriores dentro de la fuente histórica; no simula completamente una predicción al inicio de una causa. Para ese escenario habría que reconstruir qué información estaba disponible en cada fecha

In [5]:
corte_validacion = datos["fecha_fin"].iloc[int(len(datos) * 0.60)]

corte_prueba = datos["fecha_fin"].iloc[int(len(datos) * 0.80)]

entrenamiento = datos.loc[datos["fecha_fin"] < corte_validacion]

validacion = datos.loc[
    (datos["fecha_fin"] >= corte_validacion)
    & (datos["fecha_fin"] < corte_prueba)
]

prueba = datos.loc[datos["fecha_fin"] >= corte_prueba]

if min(
    len(entrenamiento),
    len(validacion),
    len(prueba),
) < 20:
    raise ValueError("Alguna partición tiene muy pocos registros")



In [6]:
print("Entrenamiento:", len(entrenamiento))
print("Validación:", len(validacion))
print("Prueba:", len(prueba))
assert entrenamiento.fecha_fin.max() < validacion.fecha_fin.min()
assert validacion.fecha_fin.max() < prueba.fecha_fin.min()

Entrenamiento: 5234
Validación: 1762
Prueba: 1756


## 4. Entrenamiento y comparación
Comparamos dos algoritmos de regresión, RandomForest y ExtraTrees, con una referencia que siempre predice la mediana. Elegimos el menor MAE en validación: se interpreta directamente en días y penaliza menos los errores extremos que una métrica cuadrática.

El OneHotEncoder se ajusta dentro del pipeline, usando solo los datos de entrenamiento en esta etapa. No estandarizamos porque estos modelos de árboles no lo necesitan. Usamos 120 árboles, hojas con al menos cinco casos y criterio de error absoluto. En esta primera versión utilizamos parámetros fijos y comparamos los algoritmos en validación. La búsqueda de hiperparámetros queda como una posible mejora

In [7]:
candidatos = {
    "BaseMediana": DummyRegressor(
        strategy="median",
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=120,
        criterion="absolute_error",
        min_samples_leaf=5,
        random_state=42,
        n_jobs=2,
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=120,
        criterion="absolute_error",
        min_samples_leaf=5,
        random_state=42,
        n_jobs=2,
    ),
}

resultados_validacion = {}

print(
    "Comparando modelos en validación...",
    flush=True,
)

for nombre, estimador in candidatos.items():
    pipeline = crear_pipeline(estimador)

    pipeline.fit(
        entrenamiento[VARIABLES],
        entrenamiento[OBJETIVO],
    )

    prediccion = pipeline.predict(
        validacion[VARIABLES]
    )

    mae = float(
        mean_absolute_error(
            validacion[OBJETIVO],
            prediccion,
        )
    )

    resultados_validacion[nombre] = mae

    print(
        f"{nombre}: MAE = {mae:.2f} días",
        flush=True,
    )

elegido = min(
    resultados_validacion,
    key=resultados_validacion.get,
)



Comparando modelos en validación...
BaseMediana: MAE = 407.44 días
RandomForest: MAE = 356.24 días
ExtraTrees: MAE = 363.62 días


## 5. Evaluación final
Una vez seleccionado el algoritmo con validación, lo reajustamos con entrenamiento y validación juntos. Recalculamos la referencia sobre ese mismo conjunto y reservamos prueba para la evaluación final

In [8]:
ajuste = pd.concat(
    [entrenamiento, validacion],
    ignore_index=True,
)

modelo = crear_pipeline(
    candidatos[elegido]
)

modelo.fit(
    ajuste[VARIABLES],
    ajuste[OBJETIVO],
)

base = DummyRegressor(
    strategy="median",
)

base.fit(
    ajuste[VARIABLES],
    ajuste[OBJETIVO],
)

predicciones = modelo.predict(
    prueba[VARIABLES]
)

predicciones_base = base.predict(
    prueba[VARIABLES]
)

mae_modelo = float(
    mean_absolute_error(
        prueba[OBJETIVO],
        predicciones,
    )
)

mae_base = float(
    mean_absolute_error(
        prueba[OBJETIVO],
        predicciones_base,
    )
)

mejora = (
    100 * (mae_base - mae_modelo) / mae_base
    if mae_base > 0
    else None
)



## 6. Métricas y alcance
Guardamos los resultados y las versiones. La cobertura de categorías indica cuántos casos de prueba tienen valores conocidos individualmente en las tres variables; no es accuracy ni cobertura de combinaciones exactas

In [13]:
categorias = {
    variable: sorted(
        ajuste[variable].unique().tolist()
    )
    for variable in VARIABLES
}

categorias_conocidas = pd.Series(
    True,
    index=prueba.index,
)

for variable in VARIABLES:
    categorias_conocidas &= (
        prueba[variable].isin(
            categorias[variable]
        )
    )

cobertura_categorias = float(
    categorias_conocidas.mean() * 100
)

metricas = {
    "modelo_elegido": elegido,
    "causas_totales": len(datos),

    "particion": {
        "entrenamiento": len(entrenamiento),
        "validacion": len(validacion),
        "prueba": len(prueba),
        "inicio_validacion": (
            corte_validacion.date().isoformat()
        ),
        "inicio_prueba": (
            corte_prueba.date().isoformat()
        ),
        "criterio": (
            "Separación temporal por finalización. "
            "Selección en validación y evaluación "
            "final en prueba."
        ),
    },

    "mae_validacion": resultados_validacion,
    "mae_modelo_prueba_dias": mae_modelo,
    "mae_base_prueba_dias": mae_base,
    "mejora_mae_porcentaje": mejora,

    "cumple_mejora_10_por_ciento": bool(
        mejora is not None
        and mejora >= 10
    ),

    "cobertura_categorias_conocidas_porcentaje": (cobertura_categorias),

    "definicion_cobertura_categorias": (
        "Porcentaje de causas de prueba cuyos "
        "valores de localidad, juzgado y objeto "
        "aparecieron individualmente en ajuste "
        "No exige antecedentes de la combinación "
        "exacta ni garantiza precisión"
    ),

    "periodo_finalizacion": {
        "desde": (
            datos["fecha_fin"]
            .min()
            .date()
            .isoformat()
        ),
        "hasta": (
            datos["fecha_fin"]
            .max()
            .date()
            .isoformat()
        ),
    },

    "variables": VARIABLES,

    "versiones": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },

    "limitaciones": [
        "Solo datos históricos de Buenos Aires",
        "Solo causas con finalización informada",
        (
            "Excluir causas sin cierre puede sesgar "
            "las duraciones observadas"
        ),
        (
            "La evaluación no demuestra capacidad "
            "para estimar plazos actuales"
        ),
        (
            "Se estima duración total histórica, "
            "no tiempo restante de causas abiertas"
        ),
    ],
}



## 7. Persistencia del modelo
Guardamos el pipeline junto con las variables, categorías, métricas y antecedentes del conjunto de ajuste. El formato conserva el contrato "judicial-duracion-v1" que utiliza la API existente. El archivo puede cargarse y predecir sin volver a entrenar

In [14]:
paquete = {
    "tipo": "judicial-duracion-v1",
    "modelo": modelo,
    "variables": VARIABLES,
    "categorias": categorias,
    "metricas": metricas,
    "antecedentes": (
        ajuste[VARIABLES + [OBJETIVO]]
        .to_dict(orient="records")
    ),
}

joblib.dump(
    paquete,
    BASE_DIR / "modelo_judicial.joblib",
)

(BASE_DIR / "metricas_judiciales.json").write_text(
    json.dumps(
        metricas,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

evaluacion = prueba[
    CLAVE_CAUSA + ["fecha_fin", OBJETIVO]
].copy()

evaluacion["prediccion_modelo_dias"] = (
    predicciones
)

evaluacion["prediccion_base_dias"] = (
    predicciones_base
)

evaluacion["categorias_conocidas"] = (
    categorias_conocidas
)

evaluacion.to_csv(
    BASE_DIR / "predicciones_prueba.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "\nRESULTADO DEL MODELO JUDICIAL\n"
)

print(
    json.dumps(
        metricas,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    f"\nModelo guardado en {BASE_DIR / 'modelo_judicial.joblib'}",
    flush=True,
)



RESULTADO DEL MODELO JUDICIAL

{
  "modelo_elegido": "RandomForest",
  "causas_totales": 8752,
  "particion": {
    "entrenamiento": 5234,
    "validacion": 1762,
    "prueba": 1756,
    "inicio_validacion": "2018-10-25",
    "inicio_prueba": "2019-04-23",
    "criterio": "Separación temporal por finalización. Selección en validación y evaluación final en prueba."
  },
  "mae_validacion": {
    "BaseMediana": 407.4381384790011,
    "RandomForest": 356.23610480514566,
    "ExtraTrees": 363.62468785471054
  },
  "mae_modelo_prueba_dias": 395.9525341685649,
  "mae_base_prueba_dias": 442.7312072892939,
  "mejora_mae_porcentaje": 10.56593082903288,
  "cumple_mejora_10_por_ciento": true,
  "cobertura_categorias_conocidas_porcentaje": 99.65831435079726,
  "definicion_cobertura_categorias": "Porcentaje de causas de prueba cuyos valores de localidad, juzgado y objeto aparecieron individualmente en ajuste No exige antecedentes de la combinación exacta ni garantiza precisión",
  "periodo_finaliz

## 8. Recuperación y prueba sin entrenamiento
Cargamos el archivo recién guardado y comparamos sus predicciones con las del modelo en memoria

In [15]:
recuperado = joblib.load(BASE_DIR / "modelo_judicial.joblib")
assert recuperado["tipo"] == "judicial-duracion-v1"
np.testing.assert_allclose(recuperado["modelo"].predict(prueba[VARIABLES]), predicciones)
ejemplo = ajuste[VARIABLES].iloc[[0]].copy()
print("Entrada:")
print(ejemplo.to_string(index=False))
print("Duración estimada (días):", round(float(recuperado["modelo"].predict(ejemplo)[0]), 1))
print("Recuperación verificada: las predicciones coinciden")

Entrada:
localidad_nombre juzgado_numero causa_objeto_litigio_descripcion
        LA PLATA              5              AMPARO (INFOREC 10)
Duración estimada (días): 157.9
Recuperación verificada: las predicciones coinciden


## 9. Lectura de resultados

In [16]:
print(f"Modelo elegido: {elegido}")
print(f"MAE de prueba: {mae_modelo:.2f} días")
print(f"MAE base: {mae_base:.2f} días")
print(f"Reducción del MAE: {mejora:.2f} %")
print(f"Reducción absoluta: {mae_base - mae_modelo:.2f} días")
print("Errores absolutos del modelo (días):")
print(pd.Series(np.abs(prueba[OBJETIVO].to_numpy() - predicciones)).describe().to_string())

Modelo elegido: RandomForest
MAE de prueba: 395.95 días
MAE base: 442.73 días
Reducción del MAE: 10.57 %
Reducción absoluta: 46.78 días
Errores absolutos del modelo (días):
count    1756.000000
mean      395.952534
std       407.237093
min         0.700000
25%       127.867708
50%       284.247917
75%       501.522917
max      2860.475000


## 10. Reflexión final y próximos pasos
El modelo elegido mejora la referencia, pero el error medio sigue siendo alto para prometer una fecha concreta de resolución. Una mejora relativa no alcanza por sí sola para asegurar utilidad en la práctica. El MAE global tampoco es un intervalo de confianza de cada consulta.

Contamos con pocas características de cada proceso y con datos de finalizaciones entre 2016 y 2019. Además, quedaron afuera causas sin fechas completas o sin finalización. Estos límites afectan la representatividad de los resultados y no permiten trasladarlos directamente a plazos actuales.

Como próximos pasos proponemos evaluar errores por localidad y objeto, revisar los grupos con pocos antecedentes e incorporar nuevas variables disponibles al momento de la consulta. Los nuevos ensayos deberían usar validación temporal; si seguimos ajustando a partir de estos resultados, necesitaremos otra evaluación independiente antes de afirmar rendimiento final.

Dejamos un pipeline persistido que reúne la transformación de las variables y el modelo. Esto permite continuar con la integración de la API y la interfaz sin repetir el entrenamiento. La aplicación existente ya consume ese formato
